# R10A — Portfolio Construction + Allocation Test

Portfolio-construction/allocation ablation only: the underlying NN v1/NN v2 BCE model and deterministic TOP2/TOP3 sector selection (R1/R7/R8/R9 `CARRY_FORWARD_BCE` + `CARRY_FORWARD_SIMPLE_SELECTION`) are never retrained or reranked -- this notebook converts those already-validated weekly signals into actual portfolio positions for the first time and compares equal-weight allocation against the paper's wins/buys/streak allocation formula. R8's dynamic ROC thresholds and R9's MC-dropout abstention are explicitly NOT used. See `docs/weekly_sector_rotation_r10a_portfolio_construction.md` for the full write-up and `r10a_portfolio.py` for the shared, unit-tested allocation-math/performance-metric implementations this notebook imports. Does not implement R10B's risk/halt filters.

In [1]:
"""
R10A full pipeline (scratchpad validation run). Once verified, transcribed
into r10a_portfolio_construction.ipynb and executed there for the committed
artifacts. Portfolio-construction/allocation ablation only: the underlying
NN v1/v2 BCE model and TOP2/TOP3 selection (R1/R7/R8/R9 CARRY_FORWARD_BCE +
CARRY_FORWARD_SIMPLE_SELECTION) are never retrained or reranked -- R10A only
converts the validated weekly signals into actual portfolio positions.
"""
import json
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler

t0 = time.time()
REPO_ROOT = Path("/Users/gavinhussey/Downloads/AtlasQuant")
FEATURE_PANEL_PATH = REPO_ROOT / "data" / "processed" / "weekly_sector_rotation" / "feature_panel.csv"
PRICES_ROOT = REPO_ROOT / "data" / "raw" / "weekly_sector_rotation" / "prices"
OUTPUTS_ROOT = REPO_ROOT / "research" / "strategies" / "weekly_sector_rotation" / "outputs"

sys.path.insert(0, str(REPO_ROOT / "research" / "strategies" / "weekly_sector_rotation"))
import r7_financial_losses as r7  # noqa: E402
import r10a_portfolio as r10a  # noqa: E402

SECTOR_ETFS = r7.SECTOR_ETFS
BENCHMARK_ETF = r7.BENCHMARK_ETF
FULL_FEATURE_COLS = r7.FULL_FEATURE_COLS

# ---------------------------------------------------------------------------

## 1. Load panel + R1 raw execution prices (next_week_first_open / next_week_final_close -- NOT adjusted close), authoritative feature set, and reproduce/verify R1's exact fold schedule.

In [2]:
# 1. Load panel + R1 target (raw execution prices), authoritative feature
#    set, fold schedule -- verbatim R1/R7/R8/R9.
# ---------------------------------------------------------------------------
date_cols = ["date", "week_end", "actual_last_trading_date", "feature_cutoff_timestamp",
             "next_week_first_trading_date", "next_week_last_trading_date"]
panel = pd.read_csv(FEATURE_PANEL_PATH, parse_dates=date_cols)
panel = panel.sort_values(["symbol", "date"]).reset_index(drop=True)
AUDIT_COLUMNS = ["week_end", "actual_last_trading_date", "week_complete", "feature_cutoff_timestamp",
                  "next_week_first_trading_date", "next_week_last_trading_date"]
FEATURE_COLUMNS = [c for c in panel.columns if c not in {"date", "symbol", *AUDIT_COLUMNS}]

r1_targets = pd.read_csv(OUTPUTS_ROOT / "r1_target_definitions_full_panel.csv", parse_dates=["date"])
# next_week_first_open / next_week_final_close are RAW (unadjusted) executable prices
# (R1's raw_long merge, NOT adjusted_close) -- next_week_open_to_close_return is
# exactly the Step 2 trade-return formula, already validated and reused verbatim.
panel = panel.merge(
    r1_targets[["date", "symbol", "target_abs_1pct_next_week", "next_week_first_open",
                "next_week_final_close", "next_week_open_to_close_return", "next_week_vti_return"]],
    on=["date", "symbol"], how="left",
)
assert set(FULL_FEATURE_COLS).issubset(set(FEATURE_COLUMNS))

md = panel.dropna(subset=FEATURE_COLUMNS + ["target_abs_1pct_next_week"]).copy()
md["label_binary"] = md["target_abs_1pct_next_week"].astype(int)
md["year"] = md["date"].dt.year
years_all = sorted(md["year"].unique())
assert years_all == list(range(2005, 2027))
test_years = r7.derive_r1_fold_schedule(years_all)
assert test_years == list(range(2008, 2027)), f"fold schedule diverges from validated R1 schedule: {test_years}"
print(f"[{time.time()-t0:.0f}s] data loaded, {len(md)} rows, test_years={test_years[0]}-{test_years[-1]}")

# ---------------------------------------------------------------------------

[0s] data loaded, 12253 rows, test_years=2008-2026


## 2. Build the VTI execution-aligned benchmark: same raw open/close convention as sector trades, using VTI's own raw daily prices and the panel's own next-week trading-day bounds (holiday-aware, not literal Monday/Friday).

In [3]:
# 2. Build the VTI execution-aligned benchmark: same raw open/close
#    convention as the sector trade_return, using VTI's own raw daily prices
#    and the panel's own (symbol-independent) next-week trading-day bounds.
# ---------------------------------------------------------------------------
week_bounds = md[["date", "next_week_first_trading_date", "next_week_last_trading_date"]].drop_duplicates(subset=["date"])
assert week_bounds["date"].is_unique

vti_daily = pd.read_csv(PRICES_ROOT / f"{BENCHMARK_ETF}.csv", parse_dates=["date"])[["date", "open", "close"]]
vti_open_lookup = vti_daily.set_index("date")["open"]
vti_close_lookup = vti_daily.set_index("date")["close"]

vti_bench_rows = []
for _, row in week_bounds.iterrows():
    entry_date, exit_date = row["next_week_first_trading_date"], row["next_week_last_trading_date"]
    if pd.isna(entry_date) or pd.isna(exit_date):
        continue
    if entry_date not in vti_open_lookup.index or exit_date not in vti_close_lookup.index:
        continue
    entry_open = vti_open_lookup.loc[entry_date]
    exit_close = vti_close_lookup.loc[exit_date]
    vti_bench_rows.append({
        "date": row["date"], "vti_entry_open": entry_open, "vti_exit_close": exit_close,
        "vti_trade_return": r10a.trade_return(entry_open, exit_close),
    })
vti_benchmark = pd.DataFrame(vti_bench_rows).set_index("date")
print(f"[{time.time()-t0:.0f}s] Step 2 (VTI execution-aligned benchmark) done, {len(vti_benchmark)} weeks.")

# ---------------------------------------------------------------------------

[0s] Step 2 (VTI execution-aligned benchmark) done, 1114 weeks.


## 3. Temporal-safety assertion (Step 1): signal timestamp strictly precedes entry, entry precedes/equals exit, for every one of the 12,253 rows.

In [4]:
# 3. Temporal-safety assertion (Step 1): signal < entry < exit for every row.
# ---------------------------------------------------------------------------
chk = md.dropna(subset=["feature_cutoff_timestamp", "next_week_first_trading_date", "next_week_last_trading_date"])
assert (chk["feature_cutoff_timestamp"] < chk["next_week_first_trading_date"]).all(), "signal must precede entry"
assert (pd.to_datetime(chk["next_week_first_trading_date"]) <= pd.to_datetime(chk["next_week_last_trading_date"])).all(), \
    "entry must precede (or equal, for 1-day weeks) exit"
print(f"[{time.time()-t0:.0f}s] Step 3 (signal < entry <= exit assertion) passed for {len(chk)} rows.")

# ---------------------------------------------------------------------------

[0s] Step 3 (signal < entry <= exit assertion) passed for 12253 rows.


## 4. Model definitions + walk-forward training -- verbatim R1/R7/R8/R9 cold-restart cadence -- then verify deterministic pooled AUC reproduces the accepted values exactly.

In [5]:
# 4. Model definitions + walk-forward training -- verbatim R1/R7/R8/R9.
# ---------------------------------------------------------------------------
NN_SEED = 20260812
HIDDEN_1, HIDDEN_2 = 16, 8
DROPOUT = 0.3
WEIGHT_DECAY = 1e-4
LEARNING_RATE = 1e-3
MAX_EPOCHS = 300
PATIENCE = 15


class SmallMLP(nn.Module):
    def __init__(self, n_features: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, HIDDEN_1), nn.ReLU(), nn.Dropout(DROPOUT),
            nn.Linear(HIDDEN_1, HIDDEN_2), nn.ReLU(), nn.Dropout(DROPOUT),
            nn.Linear(HIDDEN_2, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


V2_HIDDEN = 32
V2_DROPOUT = 0.4
V2_WEIGHT_DECAY = 1e-3
V2_LEARNING_RATE = 1e-3
V2_MAX_EPOCHS = 300
V2_PATIENCE = 15
N_SEEDS = 5
SEED_BASE = 20260812


class WideSingleLayerMLP(nn.Module):
    def __init__(self, n_features: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, V2_HIDDEN), nn.ReLU(), nn.Dropout(V2_DROPOUT),
            nn.Linear(V2_HIDDEN, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


BCE_LOSS = nn.BCEWithLogitsLoss()


def train_one_model(X_fit, y_fit, X_val, y_val, n_features, seed, v2=False):
    torch.manual_seed(seed)
    model = WideSingleLayerMLP(n_features) if v2 else SmallMLP(n_features)
    lr = V2_LEARNING_RATE if v2 else LEARNING_RATE
    wd = V2_WEIGHT_DECAY if v2 else WEIGHT_DECAY
    max_epochs = V2_MAX_EPOCHS if v2 else MAX_EPOCHS
    patience = V2_PATIENCE if v2 else PATIENCE
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    X_fit_t = torch.tensor(X_fit, dtype=torch.float32); y_fit_t = torch.tensor(y_fit, dtype=torch.float32)
    X_val_t = torch.tensor(X_val, dtype=torch.float32); y_val_t = torch.tensor(y_val, dtype=torch.float32)
    best_val_loss = float("inf"); best_state = None; epochs_without_improvement = 0
    for _ in range(max_epochs):
        model.train(); optimizer.zero_grad()
        loss = BCE_LOSS(model(X_fit_t), y_fit_t)
        loss.backward(); optimizer.step()
        model.eval()
        with torch.no_grad():
            val_loss = BCE_LOSS(model(X_val_t), y_val_t).item()
        if val_loss < best_val_loss - 1e-5:
            best_val_loss = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= patience:
                break
    model.load_state_dict(best_state); model.eval()
    return model


def walk_forward_bce(md, years_, test_years_, feature_cols, v2=False):
    oof_frames = []
    for test_year in test_years_:
        train_years_available = [y for y in years_ if y < test_year]
        val_year = train_years_available[-1]
        fit_years = train_years_available[:-1]
        train_mask = md["year"].isin(train_years_available)
        fit_mask = md["year"].isin(fit_years)
        val_mask = md["year"] == val_year
        test_mask = md["year"] == test_year
        scaler = StandardScaler().fit(md.loc[train_mask, feature_cols])
        X_fit = scaler.transform(md.loc[fit_mask, feature_cols])
        X_val = scaler.transform(md.loc[val_mask, feature_cols])
        X_test = scaler.transform(md.loc[test_mask, feature_cols])
        y_fit = md.loc[fit_mask, "label_binary"].to_numpy()
        y_val = md.loc[val_mask, "label_binary"].to_numpy()

        if v2:
            seed_predictions = []
            for seed_offset in range(N_SEEDS):
                model = train_one_model(X_fit, y_fit, X_val, y_val, len(feature_cols), SEED_BASE + seed_offset, v2=True)
                with torch.no_grad():
                    seed_predictions.append(torch.sigmoid(model(torch.tensor(X_test, dtype=torch.float32))).numpy())
            predicted_proba = np.mean(seed_predictions, axis=0)
        else:
            model = train_one_model(X_fit, y_fit, X_val, y_val, len(feature_cols), NN_SEED, v2=False)
            with torch.no_grad():
                predicted_proba = torch.sigmoid(model(torch.tensor(X_test, dtype=torch.float32))).numpy()

        fold = md.loc[test_mask, ["date", "symbol", "next_week_first_open", "next_week_final_close",
                                   "next_week_open_to_close_return", "next_week_vti_return", "label_binary"]].copy()
        fold["predicted_proba"] = predicted_proba
        oof_frames.append(fold)
    return pd.concat(oof_frames, ignore_index=True)


oof_v1 = walk_forward_bce(md, years_all, test_years, FULL_FEATURE_COLS, v2=False)
auc_v1 = roc_auc_score(oof_v1.label_binary, oof_v1.predicted_proba)
print(f"[{time.time()-t0:.0f}s] NN_v1 deterministic pooled_auc={auc_v1:.6f} (expected 0.569368)")
assert abs(auc_v1 - 0.5693678469913763) < 1e-4, "NN v1 AUC does not reproduce R1/R7/R8/R9"

oof_v2 = walk_forward_bce(md, years_all, test_years, FULL_FEATURE_COLS, v2=True)
auc_v2 = roc_auc_score(oof_v2.label_binary, oof_v2.predicted_proba)
print(f"[{time.time()-t0:.0f}s] NN_v2 deterministic pooled_auc={auc_v2:.6f} (expected 0.570832)")
assert abs(auc_v2 - 0.5708322204557499) < 1e-4, "NN v2 AUC does not reproduce R1/R7/R8/R9"
print(f"[{time.time()-t0:.0f}s] Step 4 (walk-forward training, both models) done.")

# ---------------------------------------------------------------------------

[4s] NN_v1 deterministic pooled_auc=0.569368 (expected 0.569368)


[20s] NN_v2 deterministic pooled_auc=0.570832 (expected 0.570832)
[20s] Step 4 (walk-forward training, both models) done.


## 5. Build weekly TOP-2/TOP-3 candidate tables -- pure deterministic rank, verbatim R8/R9 convention (no ROC, no MC, no backfill, no reranking).

In [6]:
# 5. Build weekly TOP-2/TOP-3 candidate tables (pure deterministic rank,
#    verbatim R8/R9 convention -- no ROC, no MC, no backfill).
# ---------------------------------------------------------------------------
OOF_BY_MODEL = {"NN_v1": oof_v1, "NN_v2": oof_v2}
K_VALUES = [2, 3]


def build_candidates(oof: pd.DataFrame, k: int) -> pd.DataFrame:
    rows = []
    for date, g in oof.groupby("date"):
        if len(g) < 11:
            continue
        ranked = g["predicted_proba"].rank(ascending=False, method="first")
        topk = g.loc[ranked <= k].copy()
        topk["rank"] = ranked.loc[ranked <= k].astype(int).to_numpy()
        rows.append(topk)
    return pd.concat(rows, ignore_index=True).sort_values(["date", "rank"]).reset_index(drop=True)


candidates = {}
for model_name, oof in OOF_BY_MODEL.items():
    for k in K_VALUES:
        candidates[(model_name, k)] = build_candidates(oof, k)
print(f"[{time.time()-t0:.0f}s] Step 5 (TOP-2/TOP-3 candidate tables) done.")

# ---------------------------------------------------------------------------

[21s] Step 5 (TOP-2/TOP-3 candidate tables) done.


## 6. Build the trade ledger for each of the 8 (model, K, allocation_method) variants: EQUAL_WEIGHT vs. PAPER_WEIGHT_NORMALIZED allocation, with the wins/buys/streak allocation-state machine evolving with NO lookahead (state used to size a trade always reflects only trades that completed strictly before that trade's own entry).

In [7]:
# 6. Build the trade ledger for each (model, K, allocation_method): weight
#    determination (and allocation-state evolution) always uses the GROSS
#    trade return -- state/weights are identical across the gross/cost/
#    integer-share settlement variants computed from this same ledger
#    (Step 12: win is actual P&L > 0; Step 9: no lookahead).
# ---------------------------------------------------------------------------
def build_trade_ledger(cand: pd.DataFrame, allocation_method: str) -> pd.DataFrame:
    tracker = r10a.AllocationStateTracker(SECTOR_ETFS)
    rows = []
    for date, week in cand.groupby("date"):
        symbols = week["symbol"].tolist()
        k = len(symbols)
        state_before = {s: tracker.state_for(s) for s in symbols}
        if allocation_method == "EQUAL_WEIGHT":
            weights = {s: r10a.equal_weight(k) for s in symbols}
        else:
            raw = {s: tracker.raw_weight_for(s) for s in symbols}
            weights = r10a.normalize_paper_weights(raw)

        for _, r in week.iterrows():
            s = r["symbol"]
            gross_ret = r["next_week_open_to_close_return"]
            rows.append({
                "date": date, "symbol": s, "rank": int(r["rank"]), "predicted_proba": r["predicted_proba"],
                "weight": weights[s], "entry_open": r["next_week_first_open"], "exit_close": r["next_week_final_close"],
                "gross_trade_return": gross_ret,
                "cost_adjusted_trade_return": r10a.apply_round_trip_cost(gross_ret),
                "vti_trade_return": r["next_week_vti_return"] if not pd.isna(r["next_week_vti_return"]) else np.nan,
                "buys_before": state_before[s].buys, "wins_before": state_before[s].wins,
                "streak_before": state_before[s].streak, "raw_weight_before": state_before[s].raw_weight(),
            })
        for s in symbols:
            g = [r for r in rows if r["date"] == date and r["symbol"] == s][0]
            tracker.record_trade(s, g["gross_trade_return"])
        for r_row in rows:
            if r_row["date"] == date:
                st = tracker.state_for(r_row["symbol"])
                r_row["buys_after"] = st.buys; r_row["wins_after"] = st.wins; r_row["streak_after"] = st.streak
    ledger = pd.DataFrame(rows)
    ledger["win"] = ledger["gross_trade_return"] > 0
    return ledger, tracker


ledgers = {}
final_trackers = {}
for (model_name, k), cand in candidates.items():
    for alloc in r10a.ALLOCATION_METHODS:
        ledger, tracker = build_trade_ledger(cand, alloc)
        ledgers[(model_name, k, alloc)] = ledger
        final_trackers[(model_name, k, alloc)] = tracker
print(f"[{time.time()-t0:.0f}s] Step 6 (trade ledgers, allocation-state evolution, all 8 variants) done.")

# ---------------------------------------------------------------------------

[26s] Step 6 (trade ledgers, allocation-state evolution, all 8 variants) done.


## 7. Settlement: (a) GROSS/fractional-share [PRIMARY], (b) COST_SENSITIVITY (10bps round-trip)/fractional, (c) INTEGER_SHARE sensitivity (gross, floor-rounded shares + residual cash).

In [8]:
# 7. Settlement: (a) GROSS fractional [PRIMARY], (b) COST_SENSITIVITY
#    fractional, (c) INTEGER_SHARE sensitivity (gross, no cost).
# ---------------------------------------------------------------------------
def settle_fractional(ledger: pd.DataFrame, return_col: str, starting_capital: float = r10a.STARTING_CAPITAL):
    weekly = ledger.groupby("date").apply(
        lambda g: (g["weight"] * g[return_col]).sum(), include_groups=False
    ).rename("portfolio_return").reset_index()
    weekly = weekly.sort_values("date").reset_index(drop=True)
    equity = r10a.equity_curve_from_returns(weekly["portfolio_return"].to_numpy(), starting_capital)
    weekly["equity_before"] = equity[:-1]
    weekly["equity_after"] = equity[1:]
    return weekly


def settle_integer_shares(ledger: pd.DataFrame, starting_capital: float = r10a.STARTING_CAPITAL):
    dates_sorted = sorted(ledger["date"].unique())
    equity = starting_capital
    rows = []
    position_rows = []
    for date in dates_sorted:
        week = ledger[ledger["date"] == date]
        end_value = 0.0
        for _, r in week.iterrows():
            allocated = equity * r["weight"]
            shares, residual = r10a.integer_shares(allocated, r["entry_open"])
            exit_value = shares * r["exit_close"]
            end_value += exit_value + residual
            position_rows.append({"date": date, "symbol": r["symbol"], "allocated_dollars": allocated,
                                   "shares": shares, "residual_cash": residual, "exit_value": exit_value})
        portfolio_return = end_value / equity - 1.0
        rows.append({"date": date, "portfolio_return": portfolio_return, "equity_before": equity, "equity_after": end_value})
        equity = end_value
    return pd.DataFrame(rows), pd.DataFrame(position_rows)


settlements_gross = {}
settlements_cost = {}
settlements_integer = {}
integer_positions = {}
for key, ledger in ledgers.items():
    settlements_gross[key] = settle_fractional(ledger, "gross_trade_return")
    settlements_cost[key] = settle_fractional(ledger, "cost_adjusted_trade_return")
    settlements_integer[key], integer_positions[key] = settle_integer_shares(ledger)
print(f"[{time.time()-t0:.0f}s] Step 7 (gross / cost / integer-share settlement, all 8 variants) done.")
for (model_name, k, alloc), weekly in settlements_gross.items():
    print(f"  {model_name} TOP{k} {alloc}: ending_equity={weekly['equity_after'].iloc[-1]:,.0f}")

# ---------------------------------------------------------------------------

[28s] Step 7 (gross / cost / integer-share settlement, all 8 variants) done.
  NN_v1 TOP2 EQUAL_WEIGHT: ending_equity=559,146
  NN_v1 TOP2 PAPER_WEIGHT_NORMALIZED: ending_equity=541,243
  NN_v1 TOP3 EQUAL_WEIGHT: ending_equity=389,266
  NN_v1 TOP3 PAPER_WEIGHT_NORMALIZED: ending_equity=393,840
  NN_v2 TOP2 EQUAL_WEIGHT: ending_equity=862,053
  NN_v2 TOP2 PAPER_WEIGHT_NORMALIZED: ending_equity=898,939
  NN_v2 TOP3 EQUAL_WEIGHT: ending_equity=492,941
  NN_v2 TOP3 PAPER_WEIGHT_NORMALIZED: ending_equity=492,204


## 8. VTI benchmark performance metrics (shared reference across all 8 variants).

In [9]:
# 8. Align VTI benchmark to each strategy's own trade-week dates; compute
#    VTI's own performance metrics once (shared across all 8 variants,
#    since it doesn't depend on model/K/allocation).
# ---------------------------------------------------------------------------
def years_elapsed_between(first_date, last_date) -> float:
    return (pd.Timestamp(last_date) - pd.Timestamp(first_date)).days / 365.25


def compute_core_metrics(weekly: pd.DataFrame, starting_capital: float = r10a.STARTING_CAPITAL) -> dict:
    returns = weekly["portfolio_return"].to_numpy()
    ending_capital = weekly["equity_after"].iloc[-1]
    years = years_elapsed_between(weekly["date"].iloc[0], weekly["date"].iloc[-1])
    cumulative_return = ending_capital / starting_capital - 1.0
    return {
        "starting_capital": starting_capital, "ending_capital": float(ending_capital),
        "cumulative_return": float(cumulative_return),
        "cagr": r10a.cagr(starting_capital, ending_capital, years),
        "annualized_volatility": r10a.annualized_volatility(returns),
        "sharpe_ratio": r10a.sharpe_ratio(returns),
        "sortino_ratio": r10a.sortino_ratio(returns),
        "max_drawdown": r10a.max_drawdown(returns),
        "calmar_ratio": r10a.calmar_ratio(r10a.cagr(starting_capital, ending_capital, years), r10a.max_drawdown(returns)),
        "mean_weekly_return": float(np.mean(returns)), "median_weekly_return": float(np.median(returns)),
        "weekly_return_std": float(np.std(returns, ddof=0)),
        "best_week": float(np.max(returns)), "worst_week": float(np.min(returns)),
        "positive_week_rate": float((returns > 0).mean()),
        "n_trade_weeks": len(weekly), "years_elapsed": years,
        "first_date": str(pd.Timestamp(weekly["date"].iloc[0]).date()),
        "last_date": str(pd.Timestamp(weekly["date"].iloc[-1]).date()),
    }


vti_weekly_for_metrics = vti_benchmark.reset_index().rename(columns={"vti_trade_return": "portfolio_return"})
vti_weekly_for_metrics["equity_after"] = r10a.equity_curve_from_returns(
    vti_weekly_for_metrics["portfolio_return"].to_numpy())[1:]
vti_core_metrics = compute_core_metrics(vti_weekly_for_metrics)
print(f"[{time.time()-t0:.0f}s] Step 8 (VTI benchmark metrics) done. VTI CAGR={vti_core_metrics['cagr']:.4f}")

# ---------------------------------------------------------------------------

[28s] Step 8 (VTI benchmark metrics) done. VTI CAGR=0.0775


## 9. Core performance metrics + vs.-VTI alpha/benchmark metrics for all 8 primary (gross, fractional) variants.

In [10]:
# 9. Core + vs-VTI metrics for all 8 primary (gross, fractional) variants.
# ---------------------------------------------------------------------------
performance_rows = []
vs_vti_rows = []
weekly_excess_store = {}
for (model_name, k, alloc), weekly in settlements_gross.items():
    core = compute_core_metrics(weekly)
    ledger = ledgers[(model_name, k, alloc)]
    n_total_trades = len(ledger)
    mean_positions = ledger.groupby("date").size().mean()
    core.update({
        "model": model_name, "K": k, "allocation_method": alloc, "variant": f"{model_name}_TOP{k}_{alloc}",
        "total_etf_trades": int(n_total_trades), "mean_positions_per_week": float(mean_positions),
        "turnover_gross_notional_over_equity_per_week": 2.0,  # full liquidate+re-enter every week (Step 24)
    })
    performance_rows.append(core)

    merged = weekly.merge(vti_weekly_for_metrics[["date", "portfolio_return"]], on="date",
                           suffixes=("", "_vti"))
    excess = merged["portfolio_return"] - merged["portfolio_return_vti"]
    weekly_excess_store[(model_name, k, alloc)] = merged.assign(excess_return=excess).set_index("date")["excess_return"]
    vs_vti_rows.append({
        "model": model_name, "K": k, "allocation_method": alloc,
        "cumulative_return_minus_vti": core["cumulative_return"] - vti_core_metrics["cumulative_return"],
        "cagr_minus_vti": core["cagr"] - vti_core_metrics["cagr"],
        "mean_weekly_excess_return": float(excess.mean()), "median_weekly_excess_return": float(excess.median()),
        "excess_return_win_rate": float((excess > 0).mean()),
        "tracking_error": r10a.tracking_error(excess.to_numpy()),
        "information_ratio": r10a.information_ratio(excess.to_numpy()),
    })

r10a_performance_summary = pd.DataFrame(performance_rows)
r10a_vs_vti_metrics = pd.DataFrame(vs_vti_rows)
print(f"[{time.time()-t0:.0f}s] Step 9 (core + vs-VTI metrics, all 8 variants) done.")
print(r10a_performance_summary[["variant", "ending_capital", "cagr", "sharpe_ratio", "max_drawdown"]].to_string())

# ---------------------------------------------------------------------------

[28s] Step 9 (core + vs-VTI metrics, all 8 variants) done.
                              variant  ending_capital      cagr  sharpe_ratio  max_drawdown
0             NN_v1_TOP2_EQUAL_WEIGHT   559145.910336  0.097116      0.526804     -0.614077
1  NN_v1_TOP2_PAPER_WEIGHT_NORMALIZED   541242.637057  0.095195      0.520665     -0.614530
2             NN_v1_TOP3_EQUAL_WEIGHT   389266.140330  0.075929      0.458558     -0.614421
3  NN_v1_TOP3_PAPER_WEIGHT_NORMALIZED   393840.274199  0.076606      0.462674     -0.602500
4             NN_v2_TOP2_EQUAL_WEIGHT   862053.212097  0.122992      0.633587     -0.519124
5  NN_v2_TOP2_PAPER_WEIGHT_NORMALIZED   898939.410563  0.125528      0.645170     -0.509378
6             NN_v2_TOP3_EQUAL_WEIGHT   492941.360136  0.089696      0.513203     -0.578286
7  NN_v2_TOP3_PAPER_WEIGHT_NORMALIZED   492203.821643  0.089609      0.513353     -0.583664


## 10. Year-by-year performance, and years-beating-VTI / total-evaluable-years.

In [11]:
# 10. Year-by-year performance (Step 19).
# ---------------------------------------------------------------------------
yearly_rows = []
for (model_name, k, alloc), weekly in settlements_gross.items():
    w = weekly.copy(); w["year"] = pd.to_datetime(w["date"]).dt.year
    vti_y = vti_weekly_for_metrics.copy(); vti_y["year"] = pd.to_datetime(vti_y["date"]).dt.year
    ledger = ledgers[(model_name, k, alloc)].copy(); ledger["year"] = pd.to_datetime(ledger["date"]).dt.year
    for year, g in w.groupby("year"):
        vti_g = vti_y[vti_y["year"] == year]
        strat_year_return = float(np.prod(1 + g["portfolio_return"]) - 1)
        vti_year_return = float(np.prod(1 + vti_g["portfolio_return"]) - 1) if len(vti_g) else np.nan
        year_equity = r10a.equity_curve_from_returns(g["portfolio_return"].to_numpy(), starting_capital=1.0)
        year_dd = r10a.max_drawdown(g["portfolio_return"].to_numpy())
        yearly_rows.append({
            "model": model_name, "K": k, "allocation_method": alloc, "year": int(year),
            "strategy_return": strat_year_return, "vti_return": vti_year_return,
            "excess_return": strat_year_return - vti_year_return if not np.isnan(vti_year_return) else np.nan,
            "max_drawdown": year_dd, "sharpe": r10a.sharpe_ratio(g["portfolio_return"].to_numpy()) if len(g) >= 2 else np.nan,
            "positive_week_rate": float((g["portfolio_return"] > 0).mean()),
            "total_etf_trades": int(len(ledger[ledger["year"] == year])),
        })
r10a_yearly_performance = pd.DataFrame(yearly_rows)

years_beat_vti_rows = []
for (model_name, k, alloc), grp in r10a_yearly_performance.groupby(["model", "K", "allocation_method"]):
    valid = grp.dropna(subset=["excess_return"])
    years_beat_vti_rows.append({
        "model": model_name, "K": k, "allocation_method": alloc,
        "years_beating_vti": int((valid["excess_return"] > 0).sum()), "total_evaluable_years": len(valid),
        "fraction_years_beating_vti": float((valid["excess_return"] > 0).mean()) if len(valid) else np.nan,
    })
r10a_years_beat_vti = pd.DataFrame(years_beat_vti_rows)
print(f"[{time.time()-t0:.0f}s] Step 10 (year-by-year performance) done.")

# ---------------------------------------------------------------------------

[28s] Step 10 (year-by-year performance) done.


## 11. Equal-weight vs. paper-weight paired comparison (Step 20) -- the primary R10A causal test.

In [12]:
# 11. Equal-weight vs paper-weight paired comparison (Step 20) -- the
#     primary R10A causal test, using paired weekly returns/excess-returns.
# ---------------------------------------------------------------------------
block_bootstrap_ci = r7.block_bootstrap_ci

equal_vs_paper_rows = []
for model_name in r10a.MODELS:
    for k in K_VALUES:
        eq = settlements_gross[(model_name, k, "EQUAL_WEIGHT")].set_index("date")
        pw = settlements_gross[(model_name, k, "PAPER_WEIGHT_NORMALIZED")].set_index("date")
        common = eq.index.intersection(pw.index)
        diff_return = (pw.loc[common, "portfolio_return"] - eq.loc[common, "portfolio_return"]).sort_index()

        eq_excess = weekly_excess_store[(model_name, k, "EQUAL_WEIGHT")]
        pw_excess = weekly_excess_store[(model_name, k, "PAPER_WEIGHT_NORMALIZED")]
        common_excess = eq_excess.index.intersection(pw_excess.index)
        diff_excess = (pw_excess.loc[common_excess] - eq_excess.loc[common_excess]).sort_index()

        eq_core = r10a_performance_summary[(r10a_performance_summary.model == model_name) &
                                            (r10a_performance_summary.K == k) &
                                            (r10a_performance_summary.allocation_method == "EQUAL_WEIGHT")].iloc[0]
        pw_core = r10a_performance_summary[(r10a_performance_summary.model == model_name) &
                                            (r10a_performance_summary.K == k) &
                                            (r10a_performance_summary.allocation_method == "PAPER_WEIGHT_NORMALIZED")].iloc[0]

        b_return = block_bootstrap_ci(diff_return)
        b_excess = block_bootstrap_ci(diff_excess)
        equal_vs_paper_rows.append({
            "model": model_name, "K": k,
            "cumulative_return_diff": pw_core["cumulative_return"] - eq_core["cumulative_return"],
            "cagr_diff": pw_core["cagr"] - eq_core["cagr"],
            "mean_weekly_return_diff_point": b_return["point_estimate"],
            "mean_weekly_return_diff_ci_lower": b_return["ci_lower"], "mean_weekly_return_diff_ci_upper": b_return["ci_upper"],
            "mean_weekly_return_diff_significant": b_return["significant"],
            "mean_weekly_excess_vti_diff_point": b_excess["point_estimate"],
            "mean_weekly_excess_vti_diff_ci_lower": b_excess["ci_lower"], "mean_weekly_excess_vti_diff_ci_upper": b_excess["ci_upper"],
            "mean_weekly_excess_vti_diff_significant": b_excess["significant"],
            "sharpe_diff": pw_core["sharpe_ratio"] - eq_core["sharpe_ratio"],
            "sortino_diff": pw_core["sortino_ratio"] - eq_core["sortino_ratio"],
            "max_drawdown_diff": pw_core["max_drawdown"] - eq_core["max_drawdown"],
            "information_ratio_paper": r10a_vs_vti_metrics[(r10a_vs_vti_metrics.model == model_name) &
                                                            (r10a_vs_vti_metrics.K == k) &
                                                            (r10a_vs_vti_metrics.allocation_method == "PAPER_WEIGHT_NORMALIZED")]["information_ratio"].iloc[0],
            "information_ratio_equal": r10a_vs_vti_metrics[(r10a_vs_vti_metrics.model == model_name) &
                                                            (r10a_vs_vti_metrics.K == k) &
                                                            (r10a_vs_vti_metrics.allocation_method == "EQUAL_WEIGHT")]["information_ratio"].iloc[0],
        })
r10a_equal_vs_paper_comparison = pd.DataFrame(equal_vs_paper_rows)
print(f"[{time.time()-t0:.0f}s] Step 11 (equal vs paper paired comparison) done.")
print(r10a_equal_vs_paper_comparison[["model", "K", "cagr_diff", "mean_weekly_return_diff_point",
                                       "mean_weekly_return_diff_significant"]].to_string())

# ---------------------------------------------------------------------------

[30s] Step 11 (equal vs paper paired comparison) done.
   model  K  cagr_diff  mean_weekly_return_diff_point  mean_weekly_return_diff_significant
0  NN_v1  2  -0.001921                      -0.000039                                False
1  NN_v1  3   0.000677                       0.000009                                False
2  NN_v2  2   0.002536                       0.000041                                False
3  NN_v2  3  -0.000088                      -0.000003                                False


## 12. Weight-distribution diagnostics (paper-weighted variants only) -- concentration/pathology check.

In [13]:
# 12. Weight-distribution diagnostics (Step 21) -- paper-weighted variants only.
# ---------------------------------------------------------------------------
weight_diag_rows = []
for (model_name, k, alloc), ledger in ledgers.items():
    if alloc != "PAPER_WEIGHT_NORMALIZED":
        continue
    largest_per_week = ledger.groupby("date")["weight"].max()
    smallest_per_week = ledger.groupby("date")["weight"].min()
    raw_weights = ledger["weight"] * 0  # placeholder overwritten below
    win_rate_component = ledger["wins_before"] / ledger["buys_before"].replace(0, np.nan)
    streak_component = ledger["streak_before"] / (ledger["wins_before"] + 1)
    weight_diag_rows.append({
        "model": model_name, "K": k,
        "mean_largest_position": float(largest_per_week.mean()), "median_largest_position": float(largest_per_week.median()),
        "p95_largest_position": float(largest_per_week.quantile(0.95)), "max_largest_position": float(largest_per_week.max()),
        "mean_smallest_position": float(smallest_per_week.mean()),
        "mean_win_rate_component": float(win_rate_component.mean(skipna=True)),
        "mean_streak_component": float(streak_component.mean()),
        "max_win_rate_component": float(win_rate_component.max(skipna=True)),
        "max_streak_component": float(streak_component.max()),
        "mean_raw_weight_before": float(ledger["raw_weight_before"].mean()),
        "max_raw_weight_before": float(ledger["raw_weight_before"].max()),
    })
r10a_weight_diagnostics = pd.DataFrame(weight_diag_rows)
print(f"[{time.time()-t0:.0f}s] Step 12 (weight-distribution diagnostics) done.")

# ---------------------------------------------------------------------------

[30s] Step 12 (weight-distribution diagnostics) done.


## 13. Per-ETF allocation-state diagnostics (paper-weighted variants only).

In [14]:
# 13. Per-ETF allocation-state diagnostics (Step 22).
# ---------------------------------------------------------------------------
per_etf_rows = []
for (model_name, k, alloc), ledger in ledgers.items():
    if alloc != "PAPER_WEIGHT_NORMALIZED":
        continue
    tracker = final_trackers[(model_name, k, alloc)]
    for symbol in SECTOR_ETFS:
        sub = ledger[ledger["symbol"] == symbol]
        final_state = tracker.state_for(symbol)
        streaks = []
        cur = 0
        for win in sub.sort_values("date")["win"]:
            cur = cur + 1 if win else 0
            streaks.append(cur)
        longest_streak = max(streaks) if streaks else 0
        per_etf_rows.append({
            "model": model_name, "K": k, "symbol": symbol,
            "total_buys": final_state.buys, "total_profitable_trades": final_state.wins,
            "empirical_win_rate": final_state.wins / final_state.buys if final_state.buys else np.nan,
            "longest_profitable_streak": longest_streak, "final_streak": final_state.streak,
            "mean_raw_score_when_selected": float(sub["raw_weight_before"].mean()) if len(sub) else np.nan,
            "max_raw_score": float(sub["raw_weight_before"].max()) if len(sub) else np.nan,
            "mean_normalized_weight_when_selected": float(sub["weight"].mean()) if len(sub) else np.nan,
        })
r10a_per_etf_allocation_state = pd.DataFrame(per_etf_rows)
print(f"[{time.time()-t0:.0f}s] Step 13 (per-ETF allocation-state diagnostics) done.")

# ---------------------------------------------------------------------------

[30s] Step 13 (per-ETF allocation-state diagnostics) done.


## 14. Transaction-cost sensitivity (10bps round trip, predeclared) and integer-share sensitivity.

In [15]:
# 14. Transaction-cost sensitivity (Step 23) + integer-share sensitivity
#     (Step 25).
# ---------------------------------------------------------------------------
cost_rows = []
for (model_name, k, alloc), weekly_cost in settlements_cost.items():
    gross_core = r10a_performance_summary[(r10a_performance_summary.model == model_name) &
                                           (r10a_performance_summary.K == k) &
                                           (r10a_performance_summary.allocation_method == alloc)].iloc[0]
    cost_core = compute_core_metrics(weekly_cost)
    cost_rows.append({
        "model": model_name, "K": k, "allocation_method": alloc,
        "gross_ending_capital": gross_core["ending_capital"], "cost_adjusted_ending_capital": cost_core["ending_capital"],
        "gross_cagr": gross_core["cagr"], "cost_adjusted_cagr": cost_core["cagr"],
        "cagr_drag_from_costs": gross_core["cagr"] - cost_core["cagr"],
        "gross_sharpe": gross_core["sharpe_ratio"], "cost_adjusted_sharpe": cost_core["sharpe_ratio"],
        "cost_bps_per_buy": r10a.COST_BPS_PER_BUY, "cost_bps_per_sell": r10a.COST_BPS_PER_SELL,
    })
r10a_cost_sensitivity = pd.DataFrame(cost_rows)
print(f"[{time.time()-t0:.0f}s] Step 14a (cost sensitivity) done.")

integer_rows = []
for (model_name, k, alloc), weekly_int in settlements_integer.items():
    gross_core = r10a_performance_summary[(r10a_performance_summary.model == model_name) &
                                           (r10a_performance_summary.K == k) &
                                           (r10a_performance_summary.allocation_method == alloc)].iloc[0]
    int_core = compute_core_metrics(weekly_int)
    pos = integer_positions[(model_name, k, alloc)]
    mean_cash_drag = float((pos["residual_cash"] / (pos["allocated_dollars"].replace(0, np.nan))).mean())
    integer_rows.append({
        "model": model_name, "K": k, "allocation_method": alloc,
        "fractional_ending_capital": gross_core["ending_capital"], "integer_share_ending_capital": int_core["ending_capital"],
        "fractional_cagr": gross_core["cagr"], "integer_share_cagr": int_core["cagr"],
        "cagr_drag_from_rounding": gross_core["cagr"] - int_core["cagr"],
        "mean_residual_cash_fraction_per_position": mean_cash_drag,
    })
r10a_integer_share_sensitivity = pd.DataFrame(integer_rows)
print(f"[{time.time()-t0:.0f}s] Step 14b (integer-share sensitivity) done.")

# ---------------------------------------------------------------------------

[30s] Step 14a (cost sensitivity) done.
[30s] Step 14b (integer-share sensitivity) done.


## 15. Standalone block bootstrap (mean weekly return, mean weekly excess return vs. VTI) for all 8 variants, plus the paper-minus-equal paired bootstrap already computed in Step 11.

In [16]:
# 15. Standalone block bootstrap (Step 26): mean weekly return, mean weekly
#     excess return vs VTI, for every one of the 8 primary variants.
# ---------------------------------------------------------------------------
bootstrap_rows = []
for (model_name, k, alloc), weekly in settlements_gross.items():
    b = block_bootstrap_ci(weekly.set_index("date")["portfolio_return"])
    bootstrap_rows.append({"model": model_name, "K": k, "allocation_method": alloc,
                            "quantity": "mean_weekly_return", **b})
    excess = weekly_excess_store[(model_name, k, alloc)]
    b = block_bootstrap_ci(excess)
    bootstrap_rows.append({"model": model_name, "K": k, "allocation_method": alloc,
                            "quantity": "mean_weekly_excess_return_vs_vti", **b})
# paper-minus-equal comparisons already computed with their own CIs in Step 11; fold them in too.
for row in equal_vs_paper_rows:
    bootstrap_rows.append({
        "model": row["model"], "K": row["K"], "allocation_method": "PAPER_MINUS_EQUAL",
        "quantity": "mean_weekly_return_diff", "point_estimate": row["mean_weekly_return_diff_point"],
        "ci_lower": row["mean_weekly_return_diff_ci_lower"], "ci_upper": row["mean_weekly_return_diff_ci_upper"],
        "n_weeks": len(settlements_gross[(row["model"], row["K"], "EQUAL_WEIGHT")]),
        "significant": row["mean_weekly_return_diff_significant"],
    })
    bootstrap_rows.append({
        "model": row["model"], "K": row["K"], "allocation_method": "PAPER_MINUS_EQUAL",
        "quantity": "mean_weekly_excess_vti_diff", "point_estimate": row["mean_weekly_excess_vti_diff_point"],
        "ci_lower": row["mean_weekly_excess_vti_diff_ci_lower"], "ci_upper": row["mean_weekly_excess_vti_diff_ci_upper"],
        "n_weeks": len(settlements_gross[(row["model"], row["K"], "EQUAL_WEIGHT")]),
        "significant": row["mean_weekly_excess_vti_diff_significant"],
    })
r10a_bootstrap_ci = pd.DataFrame(bootstrap_rows)
print(f"[{time.time()-t0:.0f}s] Step 15 (standalone + paper-vs-equal bootstrap) done.")

# ---------------------------------------------------------------------------

[33s] Step 15 (standalone + paper-vs-equal bootstrap) done.


## 16. Assemble weekly positions, trade ledger, allocation-state, and sample-audit long-format tables across all 8 variants.

In [17]:
# 16. Weekly positions, trade ledger (all variants combined), sample audit.
# ---------------------------------------------------------------------------
weekly_positions_frames = []
trade_ledger_frames = []
allocation_state_frames = []
for (model_name, k, alloc), ledger in ledgers.items():
    lg = ledger.copy(); lg.insert(0, "model", model_name); lg.insert(1, "K", k); lg.insert(2, "allocation_method", alloc)
    trade_ledger_frames.append(lg)
    pos = ledger[["date", "symbol", "rank", "weight", "predicted_proba"]].copy()
    pos.insert(0, "model", model_name); pos.insert(1, "K", k); pos.insert(2, "allocation_method", alloc)
    weekly_positions_frames.append(pos)
    state = ledger[["date", "symbol", "buys_after", "wins_after", "streak_after"]].copy()
    state.insert(0, "model", model_name); state.insert(1, "K", k); state.insert(2, "allocation_method", alloc)
    allocation_state_frames.append(state)

r10a_weekly_positions = pd.concat(weekly_positions_frames, ignore_index=True)
r10a_trade_ledger = pd.concat(trade_ledger_frames, ignore_index=True)
r10a_allocation_state = pd.concat(allocation_state_frames, ignore_index=True)

weekly_returns_frames = []
equity_curves_frames = []
for (model_name, k, alloc), weekly in settlements_gross.items():
    w = weekly.copy(); w.insert(0, "model", model_name); w.insert(1, "K", k); w.insert(2, "allocation_method", alloc)
    weekly_returns_frames.append(w)
r10a_weekly_portfolio_returns = pd.concat(weekly_returns_frames, ignore_index=True)
vti_curve_row = vti_weekly_for_metrics.copy()
vti_curve_row.insert(0, "model", "VTI_BENCHMARK"); vti_curve_row.insert(1, "K", np.nan); vti_curve_row.insert(2, "allocation_method", "N/A")
r10a_equity_curves = pd.concat([r10a_weekly_portfolio_returns, vti_curve_row], ignore_index=True)

sample_audit_rows = []
for model_name in r10a.MODELS:
    oof = OOF_BY_MODEL[model_name]
    sample_audit_rows.append({
        "model": model_name, "n_oos_rows": len(oof), "n_weeks": oof["date"].nunique(),
        "first_date": str(oof["date"].min().date()), "last_date": str(oof["date"].max().date()),
        "underlying_auc": float(roc_auc_score(oof.label_binary, oof.predicted_proba)),
        "starting_capital": r10a.STARTING_CAPITAL, "cost_bps_per_buy": r10a.COST_BPS_PER_BUY,
        "cost_bps_per_sell": r10a.COST_BPS_PER_SELL, "periods_per_year": r10a.PERIODS_PER_YEAR,
    })
r10a_sample_audit = pd.DataFrame(sample_audit_rows)
print(f"[{time.time()-t0:.0f}s] Step 16 (weekly positions / trade ledger / allocation state / sample audit) done.")

# ---------------------------------------------------------------------------

[33s] Step 16 (weekly positions / trade ledger / allocation state / sample audit) done.


## 17. Save all required R10A artifacts.

In [18]:
# 17. Save all artifacts.
# ---------------------------------------------------------------------------
r10a_weekly_positions.to_csv(OUTPUTS_ROOT / "r10a_weekly_positions.csv", index=False)
r10a_trade_ledger.to_csv(OUTPUTS_ROOT / "r10a_trade_ledger.csv", index=False)
r10a_allocation_state.to_csv(OUTPUTS_ROOT / "r10a_allocation_state.csv", index=False)
r10a_weekly_portfolio_returns.to_csv(OUTPUTS_ROOT / "r10a_weekly_portfolio_returns.csv", index=False)
r10a_equity_curves.to_csv(OUTPUTS_ROOT / "r10a_equity_curves.csv", index=False)
r10a_performance_summary.to_csv(OUTPUTS_ROOT / "r10a_performance_summary.csv", index=False)
r10a_yearly_performance.to_csv(OUTPUTS_ROOT / "r10a_yearly_performance.csv", index=False)
r10a_vs_vti_metrics.to_csv(OUTPUTS_ROOT / "r10a_vs_vti_metrics.csv", index=False)
r10a_equal_vs_paper_comparison.to_csv(OUTPUTS_ROOT / "r10a_equal_vs_paper_comparison.csv", index=False)
r10a_weight_diagnostics.to_csv(OUTPUTS_ROOT / "r10a_weight_diagnostics.csv", index=False)
r10a_per_etf_allocation_state.to_csv(OUTPUTS_ROOT / "r10a_per_etf_allocation_state.csv", index=False)
r10a_cost_sensitivity.to_csv(OUTPUTS_ROOT / "r10a_cost_sensitivity.csv", index=False)
r10a_integer_share_sensitivity.to_csv(OUTPUTS_ROOT / "r10a_integer_share_sensitivity.csv", index=False)
r10a_bootstrap_ci.to_csv(OUTPUTS_ROOT / "r10a_bootstrap_ci.csv", index=False)
r10a_sample_audit.to_csv(OUTPUTS_ROOT / "r10a_sample_audit.csv", index=False)
r10a_years_beat_vti.to_csv(OUTPUTS_ROOT / "r10a_years_beat_vti.csv", index=False)
vti_benchmark.reset_index().to_csv(OUTPUTS_ROOT / "r10a_vti_benchmark_weekly.csv", index=False)

best_variant = r10a_performance_summary.loc[r10a_performance_summary["cagr"].idxmax()]
summary = {
    "starting_capital": r10a.STARTING_CAPITAL,
    "oos_date_range": [str(oof_v1["date"].min().date()), str(oof_v1["date"].max().date())],
    "underlying_auc": {"NN_v1": float(auc_v1), "NN_v2": float(auc_v2)},
    "vti_benchmark_cagr": vti_core_metrics["cagr"], "vti_benchmark_sharpe": vti_core_metrics["sharpe_ratio"],
    "vti_benchmark_max_drawdown": vti_core_metrics["max_drawdown"],
    "variant_ending_capital": {r["variant"]: r["ending_capital"] for r in performance_rows},
    "variant_cagr": {r["variant"]: r["cagr"] for r in performance_rows},
    "variant_sharpe": {r["variant"]: r["sharpe_ratio"] for r in performance_rows},
    "variant_max_drawdown": {r["variant"]: r["max_drawdown"] for r in performance_rows},
    "best_variant_by_cagr_descriptive_only": best_variant["variant"],
    "cost_bps_round_trip": r10a.COST_BPS_PER_BUY + r10a.COST_BPS_PER_SELL,
    "runtime_seconds": time.time() - t0,
}
with open(OUTPUTS_ROOT / "r10a_run_summary.json", "w") as f:
    json.dump(summary, f, indent=2, default=str)

print(f"[{time.time()-t0:.0f}s] ALL DONE.")
print(json.dumps(summary, indent=2, default=str))

[34s] ALL DONE.
{
  "starting_capital": 100000.0,
  "oos_date_range": [
    "2008-01-04",
    "2026-07-31"
  ],
  "underlying_auc": {
    "NN_v1": 0.5693678469913763,
    "NN_v2": 0.5708322204557499
  },
  "vti_benchmark_cagr": 0.07745027188070952,
  "vti_benchmark_sharpe": 0.5222031195377523,
  "vti_benchmark_max_drawdown": -0.5532258982424447,
  "variant_ending_capital": {
    "NN_v1_TOP2_EQUAL_WEIGHT": 559145.9103363759,
    "NN_v1_TOP2_PAPER_WEIGHT_NORMALIZED": 541242.6370571743,
    "NN_v1_TOP3_EQUAL_WEIGHT": 389266.1403297695,
    "NN_v1_TOP3_PAPER_WEIGHT_NORMALIZED": 393840.2741991372,
    "NN_v2_TOP2_EQUAL_WEIGHT": 862053.212096807,
    "NN_v2_TOP2_PAPER_WEIGHT_NORMALIZED": 898939.4105633508,
    "NN_v2_TOP3_EQUAL_WEIGHT": 492941.36013561254,
    "NN_v2_TOP3_PAPER_WEIGHT_NORMALIZED": 492203.82164349995
  },
  "variant_cagr": {
    "NN_v1_TOP2_EQUAL_WEIGHT": 0.0971161951830084,
    "NN_v1_TOP2_PAPER_WEIGHT_NORMALIZED": 0.09519533714498118,
    "NN_v1_TOP3_EQUAL_WEIGHT": 0.075928